# Experiment 3: ACS 2023 target and ACS 2014 auxiliary shadows
Train the target on 100,000 California 2023 records with expanded predictors.
Train 16 diverse shadows exclusively on California 2014 auxiliary records.
Each model fits its own preprocessing on its training partition only.
The label is personal income >$50,000 in constant 2023 dollars. Apply record
ADJINC/1,000,000, then annual CPI-U(2023)/CPI-U(survey year). This adapts the
original nominal-income ACSIncome task; nominal eligibility filters are retained. Occupation and other category
codes remain native to each year; no cross-year code equivalence is assumed.
Income-derived predictors, identifiers, survey weights and allocation flags are
excluded. MIA uses logit, blind walk, entropy and cross-entropy.
Execution was authorized for this run. Ask before future cell/experiment runs.

**Protocol clarification:** Keep 100,000 target training records and the
inflation-adjusted labels. Shadow and target perturbation calls remain
identical to the original 50,000-record experiment: commuting fields are
categorical in shadow walks and numeric in target walks. Earlier statements
that the shadow protocol had been changed were incorrect. No rerun is needed
for this clarification; existing metrics already use the original protocol.


In [1]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from src.data import load_acs_income, split_acs_data, subset_target_training
from src.expanded_data import (
    load_expanded_acs, preprocess_expanded_partitions, expanded_feature_inventory,
    reference_split_config, EXPANDED_FEATURES, EXCLUDED, load_expanded_year,
)
from src.training import train_target_model, evaluate_model
from src.exp3_checkpoints import cached_target_train, cached_expanded_shadows
from src.shadow_models import load_adult_data, train_shadow_models, summarize_shadow_models
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import (collect_shadow_paper_features, collect_target_paper_features,
                                       make_probability_oracle)

torch.set_num_threads(4)
SEED = 42
TRAIN_SIZE = 100_000
INCOME_BASE_YEAR = 2023
RUN_DIR = Path('artifacts/exp_3_2023_2014_100000_real2023')
RUN_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_CONFIG = reference_split_config()
assert SPLIT_CONFIG['seed'] == SEED, 'Use the reference seed consistently.'
TRAIN_CONFIG = dict(hidden_dims=(512, 256, 128), dropout=0.0, epochs=100,
    batch_size=256, learning_rate=1e-3, weight_decay=0.0, seed=SEED,
    device='cpu', early_stopping=False, restore_best=False, monitor_train=True)
WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
                   batch_size=512, seed=SEED)
SHADOW_RECORDS_PER_CLASS = 2000
N_SHADOW_MODELS = 16
# Native commuting units; all other added codes are categorical in the walk.
EXPANDED_NUMERIC_WALK = {'JWMNP': (2.0, 0, 200), 'JWRIP': (1.0, 1, 10)}
print('Training size:', TRAIN_SIZE, '| Input fields:', len(EXPANDED_FEATURES))
print('Reference split configuration:', SPLIT_CONFIG)


Training size: 100000 | Input fields: 48
Reference split configuration: {'validation_size': 10000, 'test_size': 10000, 'mia_nonmember_size': 10000, 'mia_member_size': 10000, 'seed': 42}


## Load 2023 target records and make stratified partitions
Split sizes and seed follow Experiment 2, applied to the new 2023 records.


In [2]:
expanded_source = load_expanded_year(2023, income_base_year=INCOME_BASE_YEAR)
reference_partitions = split_acs_data(expanded_source, **SPLIT_CONFIG)
expanded_base_partitions = reference_partitions
expanded_partitions = subset_target_training(
    expanded_base_partitions, train_size=TRAIN_SIZE, seed=SEED)
for name in ['validation', 'test', 'mia_members', 'mia_nonmembers']:
    np.testing.assert_array_equal(expanded_partitions[name].index, reference_partitions[name].index)
assert set(expanded_partitions['mia_members'].index).issubset(expanded_partitions['train'].index)
for name in ['validation', 'test', 'mia_nonmembers']:
    assert set(expanded_partitions[name].index).isdisjoint(expanded_partitions['train'].index)
print('Target income adjustment:', expanded_source.attrs['income_adjustment'])
print('Eligible records:', len(expanded_source))
print('Unused records:', len(expanded_source) - TRAIN_SIZE
      - sum(len(expanded_partitions[name]) for name in ['validation', 'test', 'mia_nonmembers']))
display(pd.DataFrame({name: {'records': len(part), 'positive_rate': part['income_gt_50k'].mean()}
    for name, part in expanded_partitions.items()}).T)
feature_inventory = expanded_feature_inventory(expanded_partitions['train'])
display(feature_inventory)


Target income adjustment: {'base_year': 2023, 'survey_year': 2023, 'label_threshold': 50000, 'adjinc_divisor': 1000000, 'cpi_series': 'CPI-U US city average all items, annual NSA', 'cpi_values': {2014: 236.736, 2023: 304.702}, 'cpi_ratio': 1.0, 'cpi_source': 'https://www.bls.gov/cpi/tables/supplemental-files/historical-cpi-u-202401.pdf', 'acs_source': 'https://www2.census.gov/programs-surveys/acs/tech_docs/pums/accuracy/2023AccuracyPUMS.pdf', 'eligibility': 'Original nominal ACSIncome eligibility retained', 'labels_changed_from_nominal': 5642}
Eligible records: 203278
Unused records: 73278


,records,positive_rate
validation,10000.0,0.5262
test,10000.0,0.5262
mia_nonmembers,10000.0,0.5262
train,100000.0,0.5262
mia_members,10000.0,0.5262


,group,type,unique_nonmissing,missing_fraction
feature,,,,
AGEP,original_ACSIncome,numeric,75,0.00000
COW,original_ACSIncome,categorical,8,0.00000
SCHL,original_ACSIncome,categorical,24,0.00000
MAR,original_ACSIncome,categorical,5,0.00000
OCCP,original_ACSIncome,categorical,528,0.00000
POBP,original_ACSIncome,categorical,219,0.00000
WKHP,original_ACSIncome,numeric,97,0.00000
SEX,original_ACSIncome,categorical,2,0.00000
RAC1P,original_ACSIncome,categorical,9,0.00000


## Preprocess using training records only
The output is sparse; the MLP densifies only the current batch. Cardinality
counts above refer to the original fields, not one-hot dimensions.


In [3]:
expanded_preprocessor, expanded_arrays = preprocess_expanded_partitions(expanded_partitions)
print('Encoded input dimensions:', expanded_arrays['train']['X'].shape[1])
print('Training CSR storage (MiB):', round(sum(
    value.nbytes for value in [expanded_arrays['train']['X'].data,
    expanded_arrays['train']['X'].indices, expanded_arrays['train']['X'].indptr]) / 2**20, 2))
pd.DataFrame({name: {'rows': values['X'].shape[0], 'encoded_features': values['X'].shape[1]}
    for name, values in expanded_arrays.items()}).T


Encoded input dimensions: 2680
Training CSR storage (MiB): 38.53


,rows,encoded_features
validation,10000,2680
test,10000,2680
mia_nonmembers,10000,2680
train,100000,2680
mia_members,10000,2680


## Train the target
Fixed 100 epochs, final weights retained. Validation is monitored but does not
select an epoch in this deliberately overfit experimental condition.


In [4]:
target_started = time.monotonic()
expanded_model, expanded_history, expanded_metadata = cached_target_train(
    expanded_arrays, TRAIN_CONFIG, RUN_DIR)
target_training_seconds = time.monotonic() - target_started
print('Target training seconds:', round(target_training_seconds, 1))
display(pd.DataFrame(expanded_history).iloc[::10])


Epoch 001: train loss=0.4129, validation loss=0.3925, validation accuracy=0.8216


Epoch 002: train loss=0.3747, validation loss=0.3943, validation accuracy=0.8216


Epoch 003: train loss=0.3532, validation loss=0.3908, validation accuracy=0.8285


Epoch 004: train loss=0.3198, validation loss=0.4018, validation accuracy=0.8202


Epoch 005: train loss=0.2739, validation loss=0.4518, validation accuracy=0.8128


Epoch 006: train loss=0.2220, validation loss=0.5305, validation accuracy=0.8048


... (92 outputs left out)

Epoch 099: train loss=0.0032, validation loss=1.8319, validation accuracy=0.7825


Epoch 100: train loss=0.0026, validation loss=2.0068, validation accuracy=0.7972
Completed target checkpoint saved.


Target training seconds: 976.1


,epoch,train_loss,validation_accuracy,validation_roc_auc,validation_log_loss,train_eval_accuracy,train_eval_roc_auc,train_eval_log_loss
0,1,0.412905,0.8216,0.903042,0.392528,0.83603,0.915577,0.369549
10,11,0.058922,0.7969,0.867735,1.169548,0.98507,0.999024,0.041701
20,21,0.018754,0.7928,0.867307,1.472236,0.99528,0.999916,0.013019
30,31,0.011018,0.7957,0.868210,1.694887,0.99715,0.999957,0.007988
40,41,0.007492,0.7966,0.870959,1.695979,0.99638,0.999920,0.010569
50,51,0.005807,0.7975,0.868140,1.810647,0.99799,0.999978,0.005608
60,61,0.006117,0.7978,0.870895,1.800902,0.99924,0.999995,0.002480
70,71,0.004849,0.7985,0.872705,1.845603,0.99954,0.999999,0.001344
80,81,0.005007,0.7969,0.869119,1.699073,0.99896,0.999994,0.003210
90,91,0.003171,0.7959,0.866001,2.047964,0.99939,0.999998,0.001781


## Income-prediction evaluation
These are target income metrics, not membership metrics. The test set was not
used for model selection. Do not choose target settings using these results.


In [5]:
expanded_metrics = {name: evaluate_model(expanded_model, expanded_arrays[name]['X'],
    expanded_arrays[name]['y']) for name in ['train', 'validation', 'test']}
display(pd.DataFrame(expanded_metrics).T)
print('Train-test accuracy gap (percentage points):',
      100 * (expanded_metrics['train']['accuracy'] - expanded_metrics['test']['accuracy']))


,accuracy,roc_auc,log_loss
train,0.99975,0.999999,0.000835
validation,0.79720,0.869308,2.006771
test,0.79570,0.867252,2.057204


Train-test accuracy gap (percentage points): 20.40500000000001


## 16 diverse expanded ACS 2014 shadow models
Diverse algorithms/settings and 40/20/40 splits follow the
restored Experiment 2 attack. Each preprocessor fits its own training records.
These shadows do not assume knowledge of the target architecture.
Cycle the five algorithms using unique seeds and splits. Models may reuse auxiliary
records across shadows; grouped attack splits keep repeated records together.


In [6]:
acs_auxiliary = load_expanded_year(2014, income_base_year=INCOME_BASE_YEAR)
print('Auxiliary income adjustment:', acs_auxiliary.attrs['income_adjustment'])
assert set(acs_auxiliary.index).isdisjoint(expanded_source.index)
shadow_results = cached_expanded_shadows(acs_auxiliary, seed=SEED, n_models=N_SHADOW_MODELS, run_dir=RUN_DIR)
# Verify native-input queries reproduce probabilities computed during training.
for name, result in shadow_results.items():
    audit_rows = result['attack_data'].iloc[:32]
    native = acs_auxiliary.loc[audit_rows['record_id'], EXPANDED_FEATURES]
    query = make_probability_oracle(result['model'], result['preprocessor'], schema='acs')
    np.testing.assert_allclose(query(native),
        audit_rows[['p_income_le_50k', 'p_income_gt_50k']].to_numpy(),
        rtol=1e-5, atol=1e-6, err_msg=f'Query mismatch for {name}')
print('Probability oracle consistency verified for all 16 shadows.')
display(summarize_shadow_models(shadow_results))


Auxiliary income adjustment: {'base_year': 2023, 'survey_year': 2014, 'label_threshold': 50000, 'adjinc_divisor': 1000000, 'cpi_series': 'CPI-U US city average all items, annual NSA', 'cpi_values': {2014: 236.736, 2023: 304.702}, 'cpi_ratio': 1.287096174641795, 'cpi_source': 'https://www.bls.gov/cpi/tables/supplemental-files/historical-cpi-u-202401.pdf', 'acs_source': 'https://www2.census.gov/programs-surveys/acs/tech_docs/pums/accuracy/2023AccuracyPUMS.pdf', 'eligibility': 'Original nominal ACSIncome eligibility retained', 'labels_changed_from_nominal': 20825}


Training shadow 1/16: xgboost_01


Completed shadow checkpoint saved: xgboost_01


Training shadow 2/16: random_forest_02


Completed shadow checkpoint saved: random_forest_02


Training shadow 3/16: mlp_03


... (54 outputs left out)

Training shadow 16/16: xgboost_16


Completed shadow checkpoint saved: xgboost_16


Probability oracle consistency verified for all 16 shadows.


records  accuracy   roc_auc  log_loss
shadow_model           partition                                        
xgboost_01             train         73576  0.836346  0.919989  0.361593
                       validation    36789  0.829324  0.910891  0.378600
                       nonmembers    73576  0.828449  0.911667  0.377206
random_forest_02       train         73576  0.886498  0.961202  0.323128
                       validation    36789  0.820164  0.902312  0.410148
                       nonmembers    73576  0.818868  0.902494  0.409987
mlp_03                 train         73576  0.865160  0.939503  0.316747
                       validation    36789  0.833157  0.915442  0.368232
                       nonmembers    73576  0.829510  0.911408  0.377171
decision_tree_04       train         73576  0.855428  0.932121  0.327826
                       validation    36789  0.794748  0.860517  1.208724
                       nonmembers    73576  0.792541  0.858989  1.188528
logistic_regression_05 train         73576  0.838847  0.918104  0.363578
                       validation    36789  0.822583  0.904024  0.392126
                       nonmembers    73576  0.825120  0.904514  0.391678
xgboost_06             train         73576  0.838276  0.920893  0.359873
                       validation    36789  0.830819  0.914037  0.373153
                       nonmembers    73576  0.827145  0.909841  0.380324
random_forest_07       train         73576  0.888415  0.961968  0.320572
                       validation    36789  0.816847  0.900579  0.411763
                       nonmembers    73576  0.815646  0.899600  0.412992
mlp_08                 train         73576  0.866070  0.940300  0.314368
                       validation    36789  0.829351  0.911774  0.377218
                       nonmembers    73576  0.830787  0.913638  0.373227
decision_tree_09       train         73576  0.853566  0.931465  0.328316
                       validation    36789  0.793036  0.859628  1.177109
                       nonmembers    73576  0.791780  0.861032  1.165491
logistic_regression_10 train         73576  0.838290  0.917549  0.364702
                       validation    36789  0.826144  0.907067  0.386291
                       nonmembers    73576  0.822918  0.903765  0.393134
xgboost_11             train         73576  0.837189  0.919479  0.362614
                       validation    36789  0.828237  0.911799  0.376665
                       nonmembers    73576  0.829034  0.911519  0.377504
random_forest_12       train         73576  0.889719  0.962298  0.321157
                       validation    36789  0.819919  0.902838  0.409542
                       nonmembers    73576  0.818338  0.902226  0.410526
mlp_13                 train         73576  0.856026  0.934055  0.330332
                       validation    36789  0.833564  0.913988  0.373029
                       nonmembers    73576  0.828952  0.912269  0.376733
decision_tree_14       train         73576  0.855863  0.933254  0.324300
                       validation    36789  0.789285  0.858051  1.231156
                       nonmembers    73576  0.791100  0.858742  1.228562
logistic_regression_15 train         73576  0.838874  0.917507  0.364996
                       validation    36789  0.824268  0.904317  0.392314
                       nonmembers    73576  0.824644  0.905982  0.388402
xgboost_16             train         73576  0.838262  0.920755  0.360259
                       validation    36789  0.828644  0.911782  0.376640
                       nonmembers    73576  0.827349  0.910408  0.379293

## Extract shadow features and fit the random forest attack
Features: class-scaled logit, blind walk, entropy and cross-entropy. Use 2,000
members and 2,000 non-members per shadow. Attack record-ID grouping prevents
repeated 2014 records from crossing attack partitions. Select both thresholds
on auxiliary validation only; no target membership labels train this classifier.


In [7]:
shadow_attack_data, shadow_query_diagnostics = collect_shadow_paper_features(
    shadow_results, acs_auxiliary, per_class=SHADOW_RECORDS_PER_CLASS,
    config=WALK_CONFIG, seed=SEED)
display(pd.DataFrame(shadow_query_diagnostics).T)
attack_result = train_membership_classifier(
    shadow_attack_data, seed=SEED, algorithm='random_forest', feature_set='paper')
display(pd.DataFrame(attack_result['metrics']).T)


Attack feature extraction: 512/4,000


Attack feature extraction: 1,024/4,000


Attack feature extraction: 1,536/4,000


Attack feature extraction: 2,048/4,000


Attack feature extraction: 2,560/4,000


Attack feature extraction: 3,072/4,000


... (120 outputs left out)

Attack feature extraction: 3,584/4,000


Attack feature extraction: 4,000/4,000


,record_queries,max_record_queries,walk_censored_fraction,blind_walk_enabled,config
xgboost_01,119433,164000,0.559,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_02,122958,164000,0.56425,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
mlp_03,116063,164000,0.52125,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
decision_tree_04,98919,164000,0.37675,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
logistic_regression_05,116076,164000,0.52875,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
xgboost_06,119454,164000,0.562625,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_07,123503,164000,0.574375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
mlp_08,119196,164000,0.54575,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
decision_tree_09,99839,164000,0.392125,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
logistic_regression_10,116713,164000,0.533,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."


,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
train,38427.0,19244.0,0.602389,0.602315,0.645008,0.033309,0.505625,10665.0,8518.0,6761.0,12483.0,0.604945,0.020682,0.005995
validation,12775.0,6415.0,0.521331,0.521115,0.524426,0.012783,0.505625,2995.0,3365.0,2750.0,3665.0,0.604945,0.012783,0.009748
test,12798.0,6341.0,0.509064,0.509468,0.514200,0.011512,0.505625,3002.0,3455.0,2828.0,3513.0,0.604945,0.015613,0.013938


## Final MIA evaluation on the expanded target
Query the expanded native input schema, preserving categorical codes and
handling numeric commute fields in native units. The walk still generates
field-valid but potentially implausible combinations, and its dimensionality
matches the expanded auxiliary schema. Thresholds are frozen from auxiliary validation.


In [8]:
target_attack_data, target_query_diagnostics = collect_target_paper_features(
    expanded_model, expanded_preprocessor, expanded_partitions,
    config=WALK_CONFIG, seed=SEED, numeric_protocol=EXPANDED_NUMERIC_WALK)
display(pd.Series(target_query_diagnostics, name='expanded target query diagnostics'))
target_mia_metrics, target_membership_scores = evaluate_attack(
    attack_result['classifier'], target_attack_data,
    threshold=attack_result['accuracy_threshold'],
    low_fpr_threshold=attack_result['low_fpr_threshold'])
target_mia_metrics['tpr_at_accuracy_threshold'] = target_mia_metrics['tp'] / target_mia_metrics['members']
target_mia_metrics['fpr_at_accuracy_threshold'] = target_mia_metrics['fp'] / (
    target_mia_metrics['records'] - target_mia_metrics['members'])
display(pd.DataFrame({'2014 auxiliary attack test': attack_result['metrics']['test'],
                      'Expanded ACS target MIA': target_mia_metrics}).T)


Attack feature extraction: 512/20,000


Attack feature extraction: 1,024/20,000


Attack feature extraction: 1,536/20,000


Attack feature extraction: 2,048/20,000


Attack feature extraction: 2,560/20,000


Attack feature extraction: 3,072/20,000


... (32 outputs left out)

Attack feature extraction: 19,968/20,000


Attack feature extraction: 20,000/20,000


record_queries                                                       539484
max_record_queries                                                   820000
walk_censored_fraction                                             0.457075
blind_walk_enabled                                                     True
config                    {'max_steps': 20, 'repeats': 2, 'categorical_c...
Name: expanded target query diagnostics, dtype: object

,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold,tpr_at_accuracy_threshold,fpr_at_accuracy_threshold
2014 auxiliary attack test,12798.0,6341.0,0.509064,0.509468,0.514200,0.011512,0.505625,3002.0,3455.0,2828.0,3513.0,0.604945,0.015613,0.013938,NaN,NaN
Expanded ACS target MIA,20000.0,10000.0,0.559000,0.559000,0.616531,0.009200,0.505625,6366.0,3634.0,5186.0,4814.0,0.604945,0.080800,0.049000,0.4814,0.3634


## Small reproducibility artifacts
Save summaries, feature inventory and row assignments. Completed target and
shadow checkpoints are saved during training to support recovery after interruption.
Source CSVs and the other experiment notebooks remain unchanged.


In [9]:
result_dir = RUN_DIR
result_dir.mkdir(parents=True, exist_ok=True)
feature_inventory.to_csv(result_dir / 'feature_inventory.csv')
for name, part in expanded_partitions.items():
    pd.Series(part.index, name='record_id').to_csv(result_dir / f'{name}_ids.csv', index=False)
summarize_shadow_models(shadow_results).to_csv(result_dir / 'shadow_metrics.csv')
summary = dict(income_base_year=INCOME_BASE_YEAR,
    target_income_adjustment=expanded_source.attrs['income_adjustment'],
    auxiliary_income_adjustment=acs_auxiliary.attrs['income_adjustment'],
    perturbation_protocol_audit={'shadow_numeric_fields': {'AGEP': [1.0, 17, 99], 'WKHP': [2.0, 1, 99]}, 'target_extra_numeric_fields': {'JWMNP': [2.0, 0, 200], 'JWRIP': [1.0, 1, 10]}, 'shadow_commuting_fields': 'Categorical replacements, unchanged from original run', 'target_commuting_fields': 'Numeric perturbations, unchanged from original run', 'correction': 'Prior assistant statements that the shadow commuting perturbation protocol changed were incorrect. Original and current perturbation call sources are identical.'},
    walk_numeric_protocol_scope='Extra target numeric fields only',
    walk_numeric_protocol=EXPANDED_NUMERIC_WALK, execution_status='all_cells_completed',
    attack_metrics_status='oracle_consistency_verified', target_year=2023, auxiliary_year=2014, n_shadow_models=N_SHADOW_MODELS,
    shadow_seeds={name: result['seed'] for name, result in shadow_results.items()},
    original_feature_count=len(EXPANDED_FEATURES),
    encoded_feature_count=expanded_arrays['train']['X'].shape[1],
    train_size=TRAIN_SIZE, split_config=SPLIT_CONFIG, training=expanded_metadata,
    target_training_seconds=target_training_seconds, target_metrics=expanded_metrics,
    shadow_attack_metrics=attack_result['metrics'], target_mia_metrics=target_mia_metrics,
    target_query_diagnostics=target_query_diagnostics)
(result_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Summary and exact row IDs written to', result_dir)


Summary and exact row IDs written to artifacts\exp_3_2023_2014_100000_real2023
